<a href="https://colab.research.google.com/github/zbh3un/4002_proj1/blob/main/project1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install VADER sentiment package
!pip install vaderSentiment

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.0/126.0 kB 2.3 MB/s eta 0:00:00


In [ ]:
# Install emoji package for cleaning text
!pip install emoji

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 10.0 MB/s eta 0:00:00


In [ ]:
# Import core packages
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Import VADER sentiment analyzer
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

In [ ]:
# Initialize VADER sentiment analyzer object
analyzer = SentimentIntensityAnalyzer()

# Example sentence to demonstrate VADER output structure
sentence = "i hate candy"

# Compute sentiment scores for example sentence
scores = analyzer.polarity_scores(sentence)

# Display sentiment dictionary
print(scores)

{'neg': 0.649, 'neu': 0.351, 'pos': 0.0, 'compound': -0.5719}


In [ ]:
# Extract compound score from dictionary
compound_score = scores['compound']

# Classify sentiment using VADER thresholds
if compound_score >= 0.05:
    print("Positive")
elif compound_score <= -0.05:
    print("Negative")
else:
    print("Neutral")

Negative


In [ ]:
# Example text containing an emoji
text_with_emojis = "Excellent great music app 👍."

# Remove all emojis from the text using the emoji package
emoji_free_text = emoji.replace_emoji(text_with_emojis, "")

# Print cleaned text to verify emoji removal
print(emoji_free_text)

Excellent great music app .


In [ ]:
# Read in dataset
reviews_df = pd.read_csv("reviews.csv")
print(reviews_df.head())

        Time_submitted                                             Review  \
0  2022-07-09 15:00:00  Great music service, the audio is high quality...   
1  2022-07-09 14:21:22  Please ignore previous negative rating. This a...   
2  2022-07-09 13:27:32  This pop-up "Get the best Spotify experience o...   
3  2022-07-09 13:26:45    Really buggy and terrible to use as of recently   
4  2022-07-09 13:20:49  Dear Spotify why do I get songs that I didn't ...   

   Rating  Total_thumbsup Reply  
0       5               2   NaN  
1       5               1   NaN  
2       4               0   NaN  
3       1               1   NaN  
4       1               1   NaN  


In [ ]:
# Define function to compute VADER sentiment scores for all reviews
def sent_scores(col):

    # Determine number of reviews
    length = len(reviews_df[col])

    # Initialize arrays to store sentiment components
    neg = np.empty([1, length])
    neg = neg[0]
    neu = np.empty([1, length])
    neu = neu[0]
    pos = np.empty([1, length])
    pos = pos[0]
    compound = np.empty([1, length])
    compound = compound[0]

     # Loop through each review and compute sentiment scores
    for i in range(length):
        r = reviews_df[col][i]
        sentiment_scores = analyzer.polarity_scores(r)
        neg[i] = sentiment_scores["neg"]
        neu[i] = sentiment_scores["neu"]
        pos[i] = sentiment_scores["pos"]
        compound[i] = sentiment_scores["compound"]

    # Append sentiment scores to dataframe as new columns
    reviews_df[col+"_neg"] = neg
    reviews_df[col+"_neu"] = neu
    reviews_df[col+"_pos"] = pos
    reviews_df[col+"_compound"] = compound

# Apply sentiment scoring function to Review column
sent_scores("Review")

reviews_df.head()

,Time_submitted,Review,Rating,Total_thumbsup,Reply,Review_neg,Review_neu,Review_pos,Review_compound
0,2022-07-09 15:00:00,"Great music service, the audio is high quality...",5,2,NaN,0.000,0.564,0.436,0.9211
1,2022-07-09 14:21:22,Please ignore previous negative rating. This a...,5,1,NaN,0.234,0.377,0.389,0.6249
2,2022-07-09 13:27:32,"This pop-up ""Get the best Spotify experience o...",4,0,NaN,0.107,0.635,0.258,0.5859
3,2022-07-09 13:26:45,Really buggy and terrible to use as of recently,1,1,NaN,0.296,0.704,0.000,-0.5209
4,2022-07-09 13:20:49,Dear Spotify why do I get songs that I didn't ...,1,1,NaN,0.000,0.761,0.239,0.7149


In [ ]:
# Create a copy of the dataframe with the 'Reply' column removed
reviews_original = reviews_df.drop('Reply', axis=1)
reviews_original
reviews_original.to_csv('reviews_processed.csv', sep='\t', index=False)

In [ ]:
# Remove emojis from each review in the 'Review' column
reviews_cleaned = reviews_original['Review'].apply(lambda s: emoji.replace_emoji(s, replace=''))
reviews_cleaned

,Review
0,"Great music service, the audio is high quality..."
1,Please ignore previous negative rating. This a...
2,"This pop-up ""Get the best Spotify experience o..."
3,Really buggy and terrible to use as of recently
4,Dear Spotify why do I get songs that I didn't ...
...,...
61589,Even though it was communicated that lyrics fe...
61590,"Use to be sooo good back when I had it, and wh..."
61591,This app would be good if not for it taking ov...
61592,The app is good hard to navigate and won't jus...


In [ ]:
# Convert compound score (−1 to 1) to predicted 1–5 rating
def compound_to_rating(compound):
    return round((compound + 1) * 2 + 1)

# Compute VADER sentiment scores for all reviews in a column
def sent_scores(col):

     # Get number of reviews
    length = len(reviews_df[col])

    # Create empty arrays to store sentiment components
    neg = np.empty([1, length])
    neg = neg[0]
    neu = np.empty([1, length])
    neu = neu[0]
    pos = np.empty([1, length])
    pos = pos[0]
    compound = np.empty([1, length])
    compound = compound[0]

    # Loop through each review and compute sentiment
    for i in range(length):
        r = reviews_df[col][i]
        sentiment_scores = analyzer.polarity_scores(r)
        neg[i] = sentiment_scores["neg"]
        neu[i] = sentiment_scores["neu"]
        pos[i] = sentiment_scores["pos"]
        compound[i] = sentiment_scores["compound"]

    # Append sentiment columns to dataframe
    reviews_df[col+"_neg"] = neg
    reviews_df[col+"_neu"] = neu
    reviews_df[col+"_pos"] = pos
    reviews_df[col+"_compound"] = compound

# Apply sentiment scoring to Review column
sent_scores("Review")

In [ ]:
# Predict rating from compound score
reviews_df['predicted_rating'] = reviews_df['Review_compound'].apply(compound_to_rating)

# Calculate absolute difference between actual and predicted rating
reviews_df['rating_diff'] = abs(reviews_df['Rating'] - reviews_df['predicted_rating'])

# Identify exact matches
reviews_df['exact_match'] = reviews_df['Rating'] == reviews_df['predicted_rating']

# Compute performance metrics
accuracy = reviews_df['exact_match'].mean() * 100
avg_diff = reviews_df['rating_diff'].mean()

print(f"exact match accuracy: {accuracy:.2f}%")
print(f"avg rating difference: {avg_diff:.2f}")
reviews_df = reviews_df.drop(['Time_submitted', 'Reply', 'Total_thumbsup'], axis=1, errors='ignore')
reviews_df

exact match accuracy: 31.66%
avg rating difference: 1.09


,Review,Rating,Review_neg,Review_neu,Review_pos,Review_compound,predicted_rating,rating_diff,exact_match
0,"Great music service, the audio is high quality...",5,0.000,0.564,0.436,0.9211,5,0,True
1,Please ignore previous negative rating. This a...,5,0.234,0.377,0.389,0.6249,4,1,False
2,"This pop-up ""Get the best Spotify experience o...",4,0.107,0.635,0.258,0.5859,4,0,True
3,Really buggy and terrible to use as of recently,1,0.296,0.704,0.000,-0.5209,2,1,False
4,Dear Spotify why do I get songs that I didn't ...,1,0.000,0.761,0.239,0.7149,4,3,False
...,...,...,...,...,...,...,...,...,...
61589,Even though it was communicated that lyrics fe...,1,0.069,0.886,0.045,-0.2960,2,1,False
61590,"Use to be sooo good back when I had it, and wh...",1,0.018,0.706,0.275,0.9143,5,4,False
61591,This app would be good if not for it taking ov...,2,0.215,0.743,0.043,-0.9633,1,1,False
61592,The app is good hard to navigate and won't jus...,2,0.022,0.827,0.151,0.8074,5,3,False


In [ ]:
# Install scipt for visual analysis
!pip install scipy

In [ ]:

# Spotify Sentiment Analysis - Final 3 Figures

# Import packages
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr, pearsonr

# Set professional styling
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11
plt.rcParams['axes.labelsize'] = 12
plt.rcParams['axes.titlesize'] = 14


# Calculate review length in words
reviews_df['review_length'] = reviews_df['Review'].astype(str).apply(lambda x: len(x.split()))

# Remove any rows with missing data
reviews_df_clean = reviews_df.dropna(subset=['Review_compound', 'Rating', 'review_length'])

print(f"\nDataset Summary:")
print(f"Total reviews analyzed: {len(reviews_df_clean):,}")
print(f"Time period: {reviews_df_clean['Time_submitted'].min()} to {reviews_df_clean['Time_submitted'].max()}\n")

# Figure 1: Violin Plot by Length Categories
print("Creating Figure 1: Violin Plot...")
fig, ax = plt.subplots(figsize=(12, 7))

# Create length categories
def categorize_length(length):
    if length <= 10:
        return 'Very Short\n(≤10 words)'
    elif length <= 20:
        return 'Short\n(11-20 words)'
    elif length <= 40:
        return 'Medium\n(21-40 words)'
    elif length <= 80:
        return 'Long\n(41-80 words)'
    else:
        return 'Very Long\n(>80 words)'

reviews_df_clean['length_category'] = reviews_df_clean['review_length'].apply(categorize_length)

# Define order for categories
category_order = ['Very Short\n(≤10 words)', 'Short\n(11-20 words)',
                  'Medium\n(21-40 words)', 'Long\n(41-80 words)', 'Very Long\n(>80 words)']

# Create violin plot
parts = ax.violinplot([reviews_df_clean[reviews_df_clean['length_category'] == cat]['Review_compound'].values
                       for cat in category_order],
                      positions=range(len(category_order)),
                      widths=0.7,
                      showmeans=True,
                      showmedians=True)

# Color the violin plots with gradient
colors = ['#e8f4f8', '#b3d9e6', '#7cb9d4', '#4599c2', '#0e79b0']
for i, pc in enumerate(parts['bodies']):
    pc.set_facecolor(colors[i])
    pc.set_alpha(0.8)
    pc.set_edgecolor('black')
    pc.set_linewidth(1.5)

# Style the mean and median lines
parts['cmeans'].set_color('red')
parts['cmeans'].set_linewidth(2.5)
parts['cmedians'].set_color('darkblue')
parts['cmedians'].set_linewidth(2.5)

# Add reference lines
ax.axhline(y=0, color='gray', linestyle='--', alpha=0.7, linewidth=2,
           label='Neutral (0.0)', zorder=1)
ax.axhline(y=0.05, color='lightgray', linestyle=':', alpha=0.5, linewidth=1.5)
ax.axhline(y=-0.05, color='lightgray', linestyle=':', alpha=0.5, linewidth=1.5)

# Shade neutral region
ax.axhspan(-0.05, 0.05, alpha=0.1, color='yellow', zorder=0)

# Add sample size annotations
for i, cat in enumerate(category_order):
    count = len(reviews_df_clean[reviews_df_clean['length_category'] == cat])
    ax.text(i, -0.95, f'n={count:,}', ha='center', va='top',
            fontsize=9, fontweight='bold',
            bbox=dict(boxstyle='round,pad=0.4', facecolor='white',
                     edgecolor='black', alpha=0.9))

# Labels and title
ax.set_xticks(range(len(category_order)))
ax.set_xticklabels(category_order, fontsize=11)
ax.set_ylabel('VADER Compound Sentiment Score', fontsize=13, fontweight='bold')
ax.set_xlabel('Review Length Category', fontsize=13, fontweight='bold')
ax.set_title('Figure 1: Sentiment Distribution Across Review Length Categories',
             fontsize=15, fontweight='bold', pad=20)

ax.set_ylim(-1.05, 1.05)
ax.legend(loc='lower right', fontsize=11, framealpha=0.95)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('figure1_violin.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

print("✓ Figure 1 created and saved\n")

# Print mean sentiment by category
print("Mean sentiment by length category:")
for cat in category_order:
    cat_data = reviews_df_clean[reviews_df_clean['length_category'] == cat]['Review_compound']
    print(f"  {cat.replace(chr(10), ' ')}: {cat_data.mean():.4f} (n={len(cat_data):,})")
print()

# Figure 2: Distribution of VADER Compound Scores
print("Creating Figure 2: Sentiment Distribution...")
fig, ax = plt.subplots(figsize=(12, 7))

# Create histogram with custom coloring
n, bins, patches = ax.hist(reviews_df_clean['Review_compound'],
                           bins=60,
                           edgecolor='black',
                           alpha=0.75,
                           linewidth=0.8)

# Color code the histogram bars
for i, patch in enumerate(patches):
    bin_center = (bins[i] + bins[i+1]) / 2
    if bin_center < -0.05:
        patch.set_facecolor('#d73027')  # Red for negative
    elif bin_center > 0.05:
        patch.set_facecolor('#1a9850')  # Green for positive
    else:
        patch.set_facecolor('#fee090')  # Yellow for neutral

# Add reference lines and shading
ax.axvline(x=0, color='black', linestyle='--', linewidth=2.5,
           label='Neutral (0.0)', zorder=3)
ax.axvline(x=0.05, color='orange', linestyle=':', linewidth=2,
           label='Neutral band (±0.05)', zorder=3)
ax.axvline(x=-0.05, color='orange', linestyle=':', linewidth=2, zorder=3)

# Shade neutral region
ax.axvspan(-0.05, 0.05, alpha=0.15, color='yellow', zorder=0)

# Add annotations
mean_score = reviews_df_clean['Review_compound'].mean()
median_score = reviews_df_clean['Review_compound'].median()
ax.axvline(x=mean_score, color='blue', linestyle='-', linewidth=2,
           alpha=0.7, label=f'Mean ({mean_score:.3f})', zorder=3)
ax.axvline(x=median_score, color='purple', linestyle='-', linewidth=2,
           alpha=0.7, label=f'Median ({median_score:.3f})', zorder=3)

# Labels and title
ax.set_xlabel('VADER Compound Sentiment Score', fontsize=13, fontweight='bold')
ax.set_ylabel('Frequency (Number of Reviews)', fontsize=13, fontweight='bold')
ax.set_title('Figure 2: Distribution of VADER Compound Sentiment Scores',
             fontsize=15, fontweight='bold', pad=20)

ax.legend(loc='upper left', fontsize=10, framealpha=0.9)
ax.grid(True, alpha=0.3, axis='y')
ax.set_xlim(-1.05, 1.05)

plt.tight_layout()
plt.savefig('figure2_sentiment_distribution.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

print("✓ Figure 2 created and saved\n")

# Figure 3: Compound Score by Star Rating (Boxplot)
print("Creating Figure 3: Sentiment by Star Rating...")
fig, ax = plt.subplots(figsize=(12, 7))

# Prepare data for boxplot
box_data = [reviews_df_clean[reviews_df_clean['Rating'] == i]['Review_compound'].values
            for i in range(1, 6)]

# Create boxplot
positions = [1, 2, 3, 4, 5]
box_plot = ax.boxplot(box_data,
                       positions=positions,
                       tick_labels=['1 Star', '2 Stars', '3 Stars', '4 Stars', '5 Stars'],
                       patch_artist=True,
                       notch=True,
                       showmeans=True,
                       widths=0.6,
                       meanprops=dict(marker='D',
                                     markerfacecolor='darkred',
                                     markeredgecolor='black',
                                     markersize=8,
                                     label='Mean',
                                     zorder=3),
                       medianprops=dict(color='darkblue', linewidth=2.5, zorder=3),
                       flierprops=dict(marker='o', markerfacecolor='gray',
                                      markersize=4, alpha=0.5))

# Color the boxes with gradient from red to green
colors = ['#d73027', '#fc8d59', '#fee090', '#91cf60', '#1a9850']
for patch, color in zip(box_plot['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.8)
    patch.set_edgecolor('black')
    patch.set_linewidth(1.8)

# Style whiskers and caps
for element in ['whiskers', 'caps']:
    plt.setp(box_plot[element], color='black', linewidth=1.5)

# Add reference lines
ax.axhline(y=0, color='gray', linestyle='--', alpha=0.7, linewidth=2,
           label='Neutral (0.0)', zorder=1)
ax.axhline(y=0.05, color='lightgray', linestyle=':', alpha=0.5, linewidth=1.5, zorder=1)
ax.axhline(y=-0.05, color='lightgray', linestyle=':', alpha=0.5, linewidth=1.5, zorder=1)

# Shade neutral region
ax.axhspan(-0.05, 0.05, alpha=0.1, color='yellow', zorder=0)

# Add sample size annotations
for i, rating in enumerate(range(1, 6)):
    count = len(box_data[i])
    y_pos = -0.95
    ax.text(i+1, y_pos, f'n={count}', ha='center', va='top',
            fontsize=9, fontweight='bold', bbox=dict(boxstyle='round,pad=0.4',
            facecolor='white', edgecolor='black', alpha=0.8))

# Labels and title
ax.set_xlabel('Star Rating', fontsize=13, fontweight='bold')
ax.set_ylabel('VADER Compound Sentiment Score', fontsize=13, fontweight='bold')
ax.set_title('Figure 3: VADER Compound Sentiment Score by Star Rating',
             fontsize=15, fontweight='bold', pad=20)

ax.legend(loc='lower right', fontsize=10, framealpha=0.9)
ax.grid(True, alpha=0.3, axis='y')
ax.set_ylim(-1.05, 1.05)

plt.tight_layout()
plt.savefig('figure3_sentiment_by_rating.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

print("✓ Figure 3 created and saved\n")

# Comprehnesive Statistical Analysis
print("="*80)
print("COMPREHENSIVE STATISTICAL ANALYSIS FOR FIGURE INTERPRETATION")
print("="*80)

# 1. Overall Sentiment Distribution - for Figure 2 interpretation
print("\n1. OVERALL SENTIMENT DISTRIBUTION (Figure 2):")
print("   " + "-"*76)
mean_comp = reviews_df_clean['Review_compound'].mean()
median_comp = reviews_df_clean['Review_compound'].median()
std_comp = reviews_df_clean['Review_compound'].std()
skew_comp = reviews_df_clean['Review_compound'].skew()
min_comp = reviews_df_clean['Review_compound'].min()
max_comp = reviews_df_clean['Review_compound'].max()

print(f"   Mean compound score:     {mean_comp:7.4f}")
print(f"   Median compound score:   {median_comp:7.4f}")
print(f"   Standard deviation:      {std_comp:7.4f}")
print(f"   Skewness:                {skew_comp:7.4f}")
print(f"   Range:                   [{min_comp:.4f}, {max_comp:.4f}]")

# Interpret skewness
print(f"\n   Interpretation:")
if mean_comp > 0.1:
    print(f"   • Dataset is SHIFTED POSITIVE (mean = {mean_comp:.3f})")
elif mean_comp < -0.1:
    print(f"   • Dataset is SHIFTED NEGATIVE (mean = {mean_comp:.3f})")
else:
    print(f"   • Dataset is CENTERED NEAR NEUTRAL (mean = {mean_comp:.3f})")

if skew_comp < -0.5:
    print(f"   • Distribution is LEFT-SKEWED (skewness = {skew_comp:.3f})")
    print(f"     → More high sentiment scores, tail extends toward negative")
elif skew_comp > 0.5:
    print(f"   • Distribution is RIGHT-SKEWED (skewness = {skew_comp:.3f})")
    print(f"     → More low sentiment scores, tail extends toward positive")
else:
    print(f"   • Distribution is RELATIVELY SYMMETRIC (skewness = {skew_comp:.3f})")

# 2. Rating Distribution - for Figure 1 interpretation
print("\n2. RATING DISTRIBUTION:")
print("   " + "-"*76)
total = len(reviews_df_clean)
rating_counts = reviews_df_clean['Rating'].value_counts().sort_index()

print(f"   Total reviews: {total:,}\n")
for rating in range(1, 6):
    count = rating_counts.get(rating, 0)
    percentage = (count / total) * 100
    bar = "█" * int(percentage / 2)
    print(f"   {rating} Star: {count:6,} ({percentage:5.1f}%) {bar}")

# Check for skew
positive_reviews = len(reviews_df_clean[reviews_df_clean['Rating'] >= 4])
negative_reviews = len(reviews_df_clean[reviews_df_clean['Rating'] <= 2])
neutral_reviews = len(reviews_df_clean[reviews_df_clean['Rating'] == 3])

print(f"\n   Summary:")
print(f"   • Positive (4-5 stars): {positive_reviews:,} ({positive_reviews/total*100:.1f}%)")
print(f"   • Neutral (3 stars):    {neutral_reviews:,} ({neutral_reviews/total*100:.1f}%)")
print(f"   • Negative (1-2 stars): {negative_reviews:,} ({negative_reviews/total*100:.1f}%)")

if positive_reviews > 2 * negative_reviews:
    print(f"\n   ⚠ Dataset is HIGHLY SKEWED toward POSITIVE reviews")
    print(f"     (Positive/Negative ratio: {positive_reviews/max(negative_reviews,1):.1f}:1)")
elif negative_reviews > 2 * positive_reviews:
    print(f"\n   ⚠ Dataset is HIGHLY SKEWED toward NEGATIVE reviews")
    print(f"     (Negative/Positive ratio: {negative_reviews/max(positive_reviews,1):.1f}:1)")
else:
    print(f"\n   ✓ Dataset has BALANCED rating distribution")

# 3. Sentiment by Rating - for Figure 3 interpretation
print("\n3. COMPOUND SENTIMENT BY STAR RATING (Figure 3):")
print("   " + "-"*76)
print("   Rating │   Mean  │  Median │ Std Dev │  Count  │ % Total")
print("   " + "-"*76)

rating_stats = []
for rating in range(1, 6):
    rating_data = reviews_df_clean[reviews_df_clean['Rating'] == rating]['Review_compound']
    count = len(rating_data)
    percentage = (count / total) * 100
    mean_val = rating_data.mean()
    median_val = rating_data.median()
    std_val = rating_data.std()

    rating_stats.append({
        'rating': rating,
        'mean': mean_val,
        'median': median_val,
        'std': std_val,
        'count': count
    })

    print(f"   {rating} Star │ {mean_val:7.4f} │ {median_val:7.4f} │ {std_val:7.4f} │ "
          f"{count:7,} │ {percentage:6.1f}%")

print("   " + "-"*76)

# Check if sentiment increases with rating
means = [s['mean'] for s in rating_stats]
is_increasing = all(means[i] < means[i+1] for i in range(len(means)-1))

print(f"\n   Interpretation:")
if is_increasing:
    print(f"   ✓ Sentiment MONOTONICALLY INCREASES with star rating")
    print(f"     (1-star: {means[0]:.3f} → 5-star: {means[4]:.3f})")
    print(f"   ✓ This supports the core hypothesis: sentiment tracks ratings")
else:
    print(f"   ⚠ Sentiment does NOT strictly increase with rating")
    print(f"     (Check for anomalies in rating-sentiment alignment)")

# 4. Correlation Analysis - validates Figure 3
print("\n4. CORRELATION ANALYSIS:")
print("   " + "-"*76)
spearman_corr, spearman_p = spearmanr(reviews_df_clean['Rating'],
                                       reviews_df_clean['Review_compound'])
pearson_corr, pearson_p = pearsonr(reviews_df_clean['Rating'],
                                    reviews_df_clean['Review_compound'])

print(f"   Spearman rank correlation: {spearman_corr:.4f} (p < {spearman_p:.2e})")
print(f"   Pearson correlation:       {pearson_corr:.4f} (p < {pearson_p:.2e})")

print(f"\n   Interpretation:")
if spearman_p < 0.001:
    print(f"   ✓ Correlation is HIGHLY STATISTICALLY SIGNIFICANT (p < 0.001)")
else:
    print(f"   ⚠ Correlation significance: p = {spearman_p:.4f}")

if abs(spearman_corr) > 0.7:
    strength = "STRONG"
elif abs(spearman_corr) > 0.4:
    strength = "MODERATE"
elif abs(spearman_corr) > 0.2:
    strength = "WEAK"
else:
    strength = "VERY WEAK"

if spearman_corr > 0:
    direction = "positive"
else:
    direction = "negative"

print(f"   • {strength} {direction} correlation (ρ = {spearman_corr:.3f})")
print(f"   • {abs(spearman_corr)**2 * 100:.1f}% of variance in sentiment explained by rating")

# 5. Review Length Analysis - for Figure 1
print("\n5. REVIEW LENGTH ANALYSIS (Figure 1):")
print("   " + "-"*76)
mean_length = reviews_df_clean['review_length'].mean()
median_length = reviews_df_clean['review_length'].median()
std_length = reviews_df_clean['review_length'].std()
min_length = reviews_df_clean['review_length'].min()
max_length = reviews_df_clean['review_length'].max()

print(f"   Mean length:     {mean_length:6.1f} words")
print(f"   Median length:   {median_length:6.1f} words")
print(f"   Std deviation:   {std_length:6.1f} words")
print(f"   Range:           [{min_length:.0f}, {max_length:.0f}] words")

# Correlation between length and sentiment
length_corr, length_p = pearsonr(reviews_df_clean['review_length'],
                                  reviews_df_clean['Review_compound'])
print(f"\n   Length-Sentiment correlation: {length_corr:.4f} (p = {length_p:.4f})")

if abs(length_corr) > 0.1 and length_p < 0.05:
    if length_corr > 0:
        print(f"   • Longer reviews tend to be MORE POSITIVE")
    else:
        print(f"   • Longer reviews tend to be MORE NEGATIVE")
else:
    print(f"   • Review length shows MINIMAL correlation with sentiment")

# 6. Sentiment Categories
print("\n6. SENTIMENT CATEGORY BREAKDOWN:")
print("   " + "-"*76)
positive_sent = len(reviews_df_clean[reviews_df_clean['Review_compound'] > 0.05])
neutral_sent = len(reviews_df_clean[(reviews_df_clean['Review_compound'] >= -0.05) &
                                     (reviews_df_clean['Review_compound'] <= 0.05)])
negative_sent = len(reviews_df_clean[reviews_df_clean['Review_compound'] < -0.05])

print(f"   Positive sentiment (> 0.05):  {positive_sent:6,} ({positive_sent/total*100:5.1f}%)")
print(f"   Neutral sentiment (±0.05):    {neutral_sent:6,} ({neutral_sent/total*100:5.1f}%)")
print(f"   Negative sentiment (< -0.05): {negative_sent:6,} ({negative_sent/total*100:5.1f}%)")

